# Лабораторная работа 4. Линейная классификация: логистическая регрессия, метрики и SVM

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 3 |
| Опора на лекции | лекция 3: сигмоида (опр. 3.1), модель логистической регрессии (опр. 3.2), логистическая потеря (опр. 3.3), градиент $X^{\mathsf T}(h-y)$ и гессиан $X^{\mathsf T}SX$ (утв. 3.4–3.6), отступ (опр. 3.8), прямая и двойственная задачи SVM (опр. 3.9, теорема 3.11), опорные векторы, мягкий зазор (опр. 3.14), ядра (опр. 3.17); лекции 1–2: ERM, ММП, градиентный спуск |
| Трудоёмкость | 2 ч аудиторно (части 1–3) + 6 ч самостоятельно |

## Цель работы

Реализовать логистическую регрессию градиентным спуском и методом Ньютона, опираясь на формулы градиента и гессиана из лекции 3; научиться измерять качество классификации (матрица ошибок, precision/recall, ROC и PR-кривые) и понимать, какая метрика когда врёт; решить двойственную задачу SVM численно, найти опорные векторы и увидеть, что классификатор зависит только от них; применить ядровой переход и сравнить ядра.

## Что нужно сдать

Заполненный ноутбук `lab04_student.ipynb`, в котором:

1. выполнены все задания (ячейки с `# TODO`), код запускается сверху вниз без ошибок;
2. под каждым заданием заполнены ячейки **Вывод** — своими словами, не пересказ кода;
3. в конце — раздел «Итоги работы» с ответами на контрольные вопросы;
4. все графики подписаны (заголовок, оси, легенда).

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже). Отчёт с чужим вариантом не принимается.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from scipy import optimize
from sklearn.datasets import (load_breast_cancer, load_digits, load_wine,
                              make_circles, make_moons)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=4)
describe_variant(variant)

---
# Часть 1. Сигмоида и логистическая потеря

Определение 3.1: $\sigma(z) = \dfrac{1}{1 + e^{-z}}$, и ключевое тождество
$\sigma'(z) = \sigma(z)\bigl(1 - \sigma(z)\bigr)$ — именно оно даёт красивую
форму градиента в утверждении 3.4.

Определение 3.3 (потеря, $y \in \{0,1\}$):
$$
\mathcal L(a_\theta, x, y) = -y\ln g(x,\theta) - (1-y)\ln\bigl(1 - g(x,\theta)\bigr).
$$

Если перейти к разметке $y \in \{-1,+1\}$ и ввести **отступ** $M = y\,\theta^{\mathsf T}x$,
та же потеря записывается как $\ln(1 + e^{-M})$. В этой форме её удобно сравнивать
с пороговой потерей $[M < 0]$ и с кусочно-линейной hinge-потерей
$\max(0, 1 - M)$, которая появится в части 6 (SVM).

In [ ]:
def sigmoid(z):
    """Численно устойчивая сигмоида (без переполнения при больших |z|)."""
    # TODO: для z >= 0 считайте 1/(1+exp(-z)), для z < 0 -- exp(z)/(1+exp(z))
    raise NotImplementedError


# TODO: 1) проверьте тождество sigma' = sigma (1 - sigma) численно (np.gradient);
#       2) убедитесь, что sigmoid(-1000) не даёт переполнения;
#       3) постройте два графика: сигмоида с производной и три потери
#          (пороговая, логистическая, hinge) как функции отступа M.

> **Вывод.** Почему пороговую потерю $[M<0]$ не минимизируют напрямую? Какие два свойства логистической и hinge-потерь делают их пригодными для оптимизации?
>
> *(ваш ответ здесь)*

---
# Часть 2. Своя логистическая регрессия

Утверждение 3.4: для $Q(\theta) = \sum_{i=1}^{\ell}\mathcal L(a_\theta, x_i, y_i)$

$$
\nabla_\theta Q(\theta) = X^{\mathsf T}(h - y), \qquad h = \sigma(X\theta),
$$

а по утверждению 3.6 гессиан равен $\nabla^2 Q = X^{\mathsf T}SX$, где
$S = \mathrm{diag}\bigl(h_i(1-h_i)\bigr)$. Наличие гессиана позволяет применить
**метод Ньютона**:

$$
\theta^{(t+1)} = \theta^{(t)} - \bigl(X^{\mathsf T}SX\bigr)^{-1}X^{\mathsf T}(h - y),
$$

известный в статистике как IRLS (iteratively reweighted least squares).

Реализуйте оба метода. **Обязательно** проверьте градиент численно приёмом из
работы 2 — это ловит опечатку в формуле за минуту.

In [ ]:
def logloss(theta, X, y, l2=0.0):
    """Q(theta) = сумма логистических потерь + (l2/2)||theta_{1:}||^2.

    Подсказка: устойчиво считается как np.logaddexp(0, z) - y * z, где z = X @ theta.
    """
    raise NotImplementedError


def logloss_grad(theta, X, y, l2=0.0):
    """Градиент X^T (h - y) по утверждению 3.4 (свободный член не регуляризуем)."""
    raise NotImplementedError


def logloss_hess(theta, X, y, l2=0.0):
    """Гессиан X^T S X по утверждению 3.6, S = diag(h_i (1 - h_i))."""
    raise NotImplementedError


def fit_logreg_gd(X, y, eta=None, n_iter=500, l2=0.0):
    """Градиентный спуск. По умолчанию eta = 4/||X||_2^2 (константа Липшица)."""
    raise NotImplementedError


def fit_logreg_newton(X, y, n_iter=15, l2=1e-6):
    """Метод Ньютона (IRLS): theta -= H^{-1} g."""
    raise NotImplementedError


# TODO: обязательно проверьте градиент И гессиан конечными разностями
#       (гессиан -- это якобиан градиента).

### Задание 2.2. Данные варианта и сравнение со `sklearn`

Загрузите датасет из `variant["dataset"]`, при необходимости искусственно
разбалансируйте классы согласно `variant["class_imbalance"]`, отмасштабируйте
признаки и обучите обе свои реализации. Сравните с
`sklearn.linear_model.LogisticRegression` и постройте графики сходимости.

In [ ]:
# TODO: 1) загрузите датасет из variant["dataset"]:
#          breast_cancer -> load_breast_cancer()
#          wine          -> load_wine(), метка = (target == 0)
#          digits        -> load_digits(), только классы 3 и 8, метка = (target == 8)
#       2) примените искажение баланса из variant["class_imbalance"]
#          (оставьте указанную долю объектов класса 1);
#       3) разбейте на train/test со стратификацией, отмасштабируйте,
#          добавьте столбец единиц;
#       4) обучите свои ГС и Ньютон (l2=1.0), сравните с
#          LogisticRegression(C=1.0) -- сверьте и Q(theta), и сами коэффициенты;
#       5) постройте график сходимости обоих методов (Q - Q* в лог. масштабе).

> **Вывод.** Сколько итераций потребовалось методу Ньютона и сколько — градиентному спуску? Почему разница такая большая и почему тогда метод Ньютона не используют всегда?
>
> *(ваш ответ здесь)*

---
# Часть 3. Метрики качества классификации

Матрица ошибок для порога $t$:

| | предсказано 0 | предсказано 1 |
|---|---|---|
| **истинно 0** | TN | FP |
| **истинно 1** | FN | TP |

$$
\text{precision} = \frac{TP}{TP + FP}, \qquad
\text{recall} = \frac{TP}{TP + FN}, \qquad
F_1 = \frac{2\,\text{PR}}{\text{P} + \text{R}},
$$
$$
TPR = \text{recall}, \qquad FPR = \frac{FP}{FP + TN}.
$$

ROC-кривая — множество точек $(FPR(t), TPR(t))$ при всех порогах $t$;
AUC — площадь под ней. Полезная интерпретация:
**AUC равна вероятности того, что случайно взятый объект класса 1 получит
больший балл, чем случайно взятый объект класса 0.**

In [ ]:
def confusion(y_true, y_pred):
    """[[TN, FP], [FN, TP]]."""
    raise NotImplementedError


def roc_curve_manual(y_true, score):
    """Точки ROC-кривой. Подсказка: отсортируйте по убыванию балла и возьмите
    кумулятивные суммы меток -- это и есть TP(t) и FP(t)."""
    raise NotImplementedError


def auc_manual(fpr, tpr):
    """Площадь под кривой методом трапеций."""
    raise NotImplementedError


# TODO: 1) сверьте свои confusion и AUC со sklearn;
#       2) проверьте методом Монте-Карло интерпретацию AUC как вероятности
#          того, что случайный объект класса 1 получит больший балл, чем
#          случайный объект класса 0;
#       3) постройте три графика: ROC-кривая, PR-кривая (с линией доли класса 1),
#          зависимость precision/recall/F1 от порога с отметкой оптимального порога.

### Задание 3.2. Когда accuracy врёт

Сравните четыре величины: accuracy константного классификатора «всегда 0»,
accuracy обученной модели, ROC-AUC и PR-AUC (average precision) — при доле
класса 1, меняющейся от 0.5 до 0.02.

Чтобы эффект был виден в чистом виде, здесь берётся синтетическая выборка
`make_classification` с управляемой долей класса 1 и умеренной разделимостью
(`class_sep=0.9`, `flip_y=0.02`): на «слишком лёгких» данных вроде digits 3-vs-8
все метрики упираются в единицу и ничего не показывают.

In [ ]:
from sklearn.metrics import accuracy_score

from sklearn.datasets import make_classification

# TODO: напишите функцию, которая по заданной доле класса 1 порождает выборку
#       make_classification(n_samples=4000, n_features=20, n_informative=5,
#       n_redundant=3, class_sep=0.9, flip_y=0.02, weights=[1-frac, frac]),
#       обучает LogisticRegression и возвращает accuracy «всегда 0»,
#       accuracy модели, ROC-AUC и PR-AUC.
# TODO: постройте таблицу и график этих метрик для долей
#       [0.5, 0.35, 0.2, 0.1, 0.05, 0.03, 0.02].
print("основная метрика вашего варианта:", variant["main_metric"])

> **Вывод.** Какая метрика перестаёт различать модель и константу при росте дисбаланса? Чем PR-AUC отличается от ROC-AUC в этой ситуации и какую из них вы выберете для задачи поиска редкого события (1 % положительных)?
>
> *(ваш ответ здесь)*

---
# Часть 4. Линейно разделимая выборка: веса уходят в бесконечность

Если выборка линейно разделима, то у логистической потери **нет минимума**:
для любого $\theta$, разделяющего классы, увеличение $\|\theta\|$ в $c$ раз
увеличивает все отступы в $c$ раз и уменьшает $Q(\theta)$. Формально
$\inf_\theta Q(\theta) = 0$, но инфимум не достигается.

Проверьте это: обучите модель без регуляризации на разделимых данных и следите
за $\|\theta\|$. Затем добавьте $L_2$-штраф.

In [ ]:
X_sep = np.column_stack([np.ones(60), rng.normal(size=(60, 2))])
y_sep = (X_sep[:, 1] + X_sep[:, 2] > 0).astype(float)      # заведомо разделимо

# TODO: обучите логрег градиентным спуском 20000 итераций при l2 = 0 и при l2 = 1,
#       записывая ||theta|| и Q каждые 100 итераций, и постройте два графика.

> **Вывод.** Как ведёт себя $\|\theta\|$ без регуляризации и с ней? Что при этом происходит с предсказанными вероятностями и почему это плохо, даже если все ответы верны?
>
> *(ваш ответ здесь)*

---
# Часть 5. SVM: двойственная задача и опорные векторы

Теорема 3.11: двойственная задача SVM —

$$
\max_{\alpha}\ \sum_{i=1}^{\ell}\alpha_i
- \frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j (x_i^{\mathsf T}x_j)
\quad\text{при}\quad \alpha_i \ge 0,\ \ \sum_i \alpha_i y_i = 0 ,
$$

и далее $w^* = \sum_i \alpha_i^* y_i x_i$, а $b^*$ — из любого объекта
с $\alpha_i^* > 0$: $b^* = y_i - w^{*\mathsf T}x_i$.
Объекты с $\alpha_i^* > 0$ — **опорные векторы**.

Решите двойственную задачу способом из `variant["own_solver"]` (в эталоне
приведены все три) для линейно разделимой выборки.

In [ ]:
def solve_dual_slsqp(K, y, C=None):
    """Двойственная задача теоремы 3.11 через scipy.optimize.minimize (SLSQP).

    Минимизируется 0.5 a^T P a - sum(a), где P = (y y^T) * K,
    при ограничениях 0 <= a_i <= C и a^T y = 0.
    """
    raise NotImplementedError


def solve_dual_projgrad(K, y, C=None, n_iter=20_000):
    """Проекционный градиентный подъём (см. подсказку в тексте)."""
    raise NotImplementedError


def solve_dual_smo(K, y, C=1e6, n_passes=40, tol=1e-6):
    """Упрощённый SMO: оптимизация пары множителей."""
    raise NotImplementedError


def recover_wb(alpha, X, y, C=None, tol=1e-6):
    """w = sum a_i y_i x_i; b -- по объектам с 0 < a_i (< C)."""
    raise NotImplementedError


from sklearn.datasets import make_blobs
Xb, yb = make_blobs(n_samples=60, centers=2, cluster_std=1.0, random_state=RANDOM_STATE)
yb = np.where(yb == 0, -1.0, 1.0)
Xb = StandardScaler().fit_transform(Xb) * 1.2

# TODO: реализуйте решатель СВОЕГО варианта, решите двойственную задачу,
#       восстановите (w, b), посчитайте число опорных векторов и ширину полосы
#       2/||w||; сравните с SVC(kernel="linear", C=1e6).
print("решатель вашего варианта:", variant["own_solver"])

### Задание 5.2. Картинка: полоса и опорные векторы

Нарисуйте разделяющую гиперплоскость $w^{\mathsf T}x + b = 0$, границы полосы
$w^{\mathsf T}x + b = \pm 1$ и выделите опорные векторы. Затем **удалите из
выборки все неопорные объекты**, переобучите SVM и убедитесь, что решение
не изменилось.

In [ ]:
# TODO: 1) нарисуйте выборку, разделяющую прямую w^T x + b = 0, границы полосы
#          w^T x + b = +-1 и обведите опорные векторы;
#       2) переобучите SVM ТОЛЬКО на опорных векторах и сравните (w, b)
#          с решением на полной выборке.

> **Вывод.** Сколько опорных векторов получилось и изменилось ли решение после удаления остальных объектов? Какое свойство двойственной задачи это объясняет? Посмотрите на спектр множителей у итеративного решателя: почему там нет точных нулей и как это влияет на определение опорного вектора?
>
> *(ваш ответ здесь)*

---
# Часть 6. Мягкий зазор и параметр $C$

Определение 3.14: $\min \frac12\|w\|^2 + C\sum_i\xi_i$ при
$y_i(w^{\mathsf T}x_i + b) \ge 1 - \xi_i$, $\xi_i \ge 0$. В двойственной задаче
добавляется ограничение $\alpha_i \le C$.

Исключив $\xi_i = \max(0, 1 - y_i(w^{\mathsf T}x_i + b))$, получаем эквивалентную
безусловную запись — **регуляризованную минимизацию hinge-потери**:

$$
\min_{w,b}\ \frac12\|w\|^2 + C\sum_{i=1}^{\ell}\max\bigl(0,\ 1 - y_i(w^{\mathsf T}x_i + b)\bigr).
$$

Сравните её с $L_2$-регуляризованной логистической регрессией из части 2:
отличается только функция потерь (см. график из части 1).

In [ ]:
Xm, ym = make_moons(n_samples=200, noise=0.28, random_state=RANDOM_STATE)
ym = np.where(ym == 0, -1.0, 1.0)
Xm = StandardScaler().fit_transform(Xm)

# TODO: напишите функцию plot_decision(ax, predict, X, y, title), рисующую
#       области решения и линии уровня decision_function = -1, 0, +1.
# TODO: для C in [0.01, 0.1, 1, 100] обучите SVC(kernel="linear", C=C),
#       нарисуйте границы и сведите в таблицу: число опорных векторов,
#       ширину полосы 2/||w||, число ошибок на обучении.

> **Вывод.** Как число опорных векторов и ширина полосы зависят от $C$? Какому пределу соответствует $C \to \infty$, а какому — $C \to 0$? Сопоставьте $C$ с параметром регуляризации $\lambda$ из работы 3.
>
> *(ваш ответ здесь)*

---
# Часть 7. Ядра

Определение 3.17: $K(x,x') = \langle\varphi(x), \varphi(x')\rangle$.
Критерий Мерсера: $K$ — ядро тогда и только тогда, когда матрица Грама
$\|K(x_i,x_j)\|$ симметрична и положительно полуопределена для любого набора точек.

Проверьте три вещи:

1. **Явное отображение.** Для квадратичного ядра из примера лекции
   $\varphi(x) = (x_1^2, \sqrt2\,x_1x_2, x_2^2)$ убедитесь, что
   $\langle\varphi(x),\varphi(x')\rangle = (x^{\mathsf T}x')^2$.
2. **Критерий Мерсера.** Посчитайте собственные числа матриц Грама для
   полиномиального и RBF-ядер — и для функции, ядром **не** являющейся
   (например, $K(x,x') = \|x - x'\|$).
3. **Ядровой SVM.** Решите двойственную задачу со своими ядрами
   (`variant["kernels"]`) на нелинейно разделимых данных.

In [ ]:
def kernel_linear(A, B):
    raise NotImplementedError


def kernel_poly(A, B, d=3, c=1.0):
    raise NotImplementedError


def kernel_rbf(A, B, gamma=1.0):
    """Подсказка: ||a-b||^2 = ||a||^2 - 2<a,b> + ||b||^2 (работа 1, часть 1)."""
    raise NotImplementedError


def kernel_sigmoid(A, B, gamma=0.1, c=0.0):
    raise NotImplementedError


# TODO: (1) проверьте, что <phi(x), phi(x')> = (x^T x')^2 для
#           phi(x) = (x_1^2, sqrt(2) x_1 x_2, x_2^2);
#       (2) для 40 случайных точек посчитайте матрицы Грама всех четырёх ядер
#           и функции K(x,x') = ||x - x'||; по знаку минимального собственного
#           числа определите, какие из них ядра (критерий Мерсера).

### Задание 7.2. Ядровой SVM своими руками

Соберите ядровой классификатор
$a(x) = \mathrm{sign}\bigl(\sum_i \alpha_i^* y_i K(x_i, x) + b^*\bigr)$
на основе своего решателя двойственной задачи и сравните ядра из вашего варианта
на выборках `make_moons` и `make_circles`.

In [ ]:
class KernelSVM:
    """SVM с произвольным ядром: обучение через двойственную задачу.

    fit: решить двойственную задачу с матрицей K(X, X), запомнить alpha,
         восстановить b по объектам с 0 < alpha < C.
    decision_function: sum_i alpha_i y_i K(x_i, x) + b.
    """
    raise NotImplementedError


# TODO: сравните ядра из variant["kernels"] на make_moons и make_circles:
#       для каждой пары нарисуйте границу решения, выведите точность
#       и число опорных векторов.

### Задание 7.3. Параметр $\gamma$ у RBF-ядра

$K(x,x') = \exp(-\gamma\|x-x'\|^2)$, где $\gamma = 1/(2\sigma^2)$.
Постройте границы решения для $\gamma \in \{0.01, 0.1, 1, 10, 100\}$ и
проследите за точностью на обучении и на контроле.

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xm, ym, test_size=0.4, random_state=RANDOM_STATE, stratify=ym)

# TODO: для gamma in [0.01, 0.1, 1, 10, 100] обучите SVC(kernel="rbf"),
#       нарисуйте границы и сведите в таблицу точности на обучении и контроле
#       и число опорных векторов.

> **Вывод.** Что происходит с границей при $\gamma \to 0$ и при $\gamma \to \infty$? На каком $\gamma$ разрыв между обучением и контролем максимален и как это называется?
>
> *(ваш ответ здесь)*

---
# Часть 8. Своя выборка

Сравните на индивидуальной выборке логистическую регрессию и SVM с ядрами
вашего варианта по основной метрике `variant["main_metric"]`.
Если ваш вариант — регрессионный, целевую переменную бинаризуйте по медиане
(и укажите это в отчёте).

In [ ]:
from sklearn.metrics import balanced_accuracy_score, f1_score

data = load_personal(variant)
# TODO: 1) если ваш вариант регрессионный -- бинаризуйте цель по медиане
#          ОБУЧАЮЩЕЙ выборки (не всей!);
#       2) обучите логистическую регрессию и SVM со всеми ядрами вашего варианта;
#       3) сравните их по основной метрике variant["main_metric"], по accuracy
#          и по числу опорных векторов; добавьте строку с константным ответом.

> **Вывод.** Какая модель победила по вашей основной метрике? Оправдало ли ядро усложнение по сравнению с линейной моделью? Сколько объектов оказалось опорными и о чём это говорит?
>
> *(ваш ответ здесь)*

## Итоги работы

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Логистическая регрессия минимизирует $\sum_i \ln(1+e^{-M_i})$, SVM — $\frac12\|w\|^2 + C\sum_i\max(0, 1-M_i)$. Назовите два различия в поведении, которые следуют прямо из вида этих потерь.
2. Почему на линейно разделимой выборке у логистической регрессии без регуляризации нет минимума, а у SVM с жёстким зазором — есть?
3. У вас 1 000 000 объектов, из них 500 положительных. Модель даёт accuracy 0.9995 и ROC-AUC 0.97. Достаточно ли этого, чтобы внедрять модель? Что ещё нужно посмотреть?
4. Что такое опорный вектор и почему классификатор не меняется при удалении неопорных объектов? Сошлитесь на условие дополняющей нежёсткости.
5. Ядро $K(x,x') = \|x-x'\|$ не удовлетворяет критерию Мерсера. Что конкретно сломается, если подставить его в двойственную задачу?

### Домашнее задание

1. **Многоклассовое обобщение.** Реализуйте схемы «один против всех» (one-vs-rest, $M$ классификаторов) и «каждый против каждого» (one-vs-one, $M(M-1)/2$ классификаторов) поверх своей бинарной логистической регрессии. Проверьте на `load_digits` (10 классов): сравните точность, время обучения и число обучаемых моделей. Покажите на рисунке область, в которой обе схемы дают неоднозначный ответ (для one-vs-rest — где несколько классификаторов выдают +1). Сравните со `sklearn` (`multi_class='ovr'` и `SVC` с `decision_function_shape`).

2. **Полиномиальное ядро против RBF.** Сгенерируйте выборку, на которой полиномиальное ядро степени 2 **строго лучше** и линейного, и RBF при честном подборе $C$ и $\gamma$ по отложенной выборке (подсказка: класс задаётся квадратичной формой $x^{\mathsf T}Ax \ge c$). Приведите таблицу качества и объясните, почему RBF проигрывает, хотя формально его пространство признаков бесконечномерно.